<a href="https://colab.research.google.com/github/DavidSenseman/BIO1173/blob/main/ResNet_Model_Recovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ResNet Model Recovery Notebook
**This notebook is only for students who did not complete the earlier class lesson or who have accidentally deleted their model files.**

It will:
1. Mount your Google Drive.
2. Check whether the required model folders exist — creating them if not.
3. Download the missing model files directly into your Google Drive.
4. Display a final summary confirming everything is in place.

> ⚠️ If you already have both model files on your Google Drive from the earlier lesson, you do **not** need to run this notebook.

## Step 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ Google Drive mounted at /content/drive')

Mounted at /content/drive
✅ Google Drive mounted at /content/drive


## Step 2 — Check / Create Model Folders

In [2]:
import os

def check_or_create_folder(folder_name: str, base_path: str = '/content/drive/MyDrive') -> str:
    """
    Check whether `folder_name` exists inside `base_path`.
    If it does not exist, create it.
    Returns the full path to the folder.
    """
    full_path = os.path.join(base_path, folder_name)

    if os.path.isdir(full_path):
        print(f'✅ Folder already exists : {full_path}')
    else:
        os.makedirs(full_path, exist_ok=True)
        if os.path.isdir(full_path):
            print(f'📁 Folder created        : {full_path}')
        else:
            raise RuntimeError(f'❌ Failed to create folder: {full_path}')

    return full_path


# ── Check / create both model folders ────────────────────────────────────────
FOLDER_RESNET50  = check_or_create_folder('ResNet50_model_244')
FOLDER_RESNET101 = check_or_create_folder('ResNet101_model_512')

✅ Folder already exists : /content/drive/MyDrive/ResNet50_model_244
✅ Folder already exists : /content/drive/MyDrive/ResNet101_model_512


## Step 3 — Download Model Files and Verify

In [3]:
import os
import time
import requests

# ── Download configuration ────────────────────────────────────────────────────
MODELS = [
    {
        'url'        : 'https://biologicslab.co/BIO1173/data/ResNet50_model_244.pth',
        'dest_folder': FOLDER_RESNET50,
        'filename'   : 'ResNet50_model_244.pth',
        'min_size_mb': 1,
    },
    {
        'url'        : 'https://biologicslab.co/BIO1173/data/ResNet101_model_512.pth',
        'dest_folder': FOLDER_RESNET101,
        'filename'   : 'ResNet101_model_512.pth',
        'min_size_mb': 1,
    },
]


def download_model(url: str, dest_folder: str, filename: str, min_size_mb: float = 1.0) -> str:
    """
    Download a file from `url` into `dest_folder/filename` using streaming.
    Verifies the file exists and meets the minimum size threshold.
    Returns the full local path of the downloaded file.
    """
    os.makedirs(dest_folder, exist_ok=True)
    dest_path      = os.path.join(dest_folder, filename)
    min_size_bytes = int(min_size_mb * 1024 ** 2)

    print(f'⬇️  Downloading : {filename}')
    print(f'   From        : {url}')
    print(f'   To          : {dest_path}')

    start = time.time()
    try:
        response = requests.get(url, stream=True, timeout=300)
        response.raise_for_status()
        with open(dest_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
    except requests.exceptions.Timeout:
        raise RuntimeError(f'❌ Download timed out for {filename}. Try restarting the Colab runtime and running again.')
    except requests.exceptions.HTTPError as http_err:
        raise RuntimeError(f'❌ HTTP error for {filename}: {http_err}')
    except Exception as exc:
        raise RuntimeError(f'❌ Download failed for {filename}: {exc}') from exc

    elapsed = time.time() - start

    # ── Verification ──────────────────────────────────────────────────────────
    if not os.path.isfile(dest_path):
        raise FileNotFoundError(f'❌ File not found after download: {dest_path}')

    file_size = os.path.getsize(dest_path)

    if file_size == 0:
        raise ValueError(f'❌ Downloaded file is empty (0 bytes): {filename}')

    if file_size < min_size_bytes:
        raise ValueError(
            f'❌ {filename} is suspiciously small: {file_size:,} bytes '
            f'(expected ≥ {min_size_bytes:,} bytes). '
            'The server may have returned an error page instead of the model.'
        )

    print(f'   ✅ Verified  : {file_size / (1024**2):.2f} MB in {elapsed:.1f}s\n')
    return dest_path


# ── Download both models ──────────────────────────────────────────────────────
for model in MODELS:
    download_model(**model)

⬇️  Downloading : ResNet50_model_244.pth
   From        : https://biologicslab.co/BIO1173/data/ResNet50_model_244.pth
   To          : /content/drive/MyDrive/ResNet50_model_244/ResNet50_model_244.pth
   ✅ Verified  : 91.99 MB in 10.8s

⬇️  Downloading : ResNet101_model_512.pth
   From        : https://biologicslab.co/BIO1173/data/ResNet101_model_512.pth
   To          : /content/drive/MyDrive/ResNet101_model_512/ResNet101_model_512.pth
   ✅ Verified  : 164.74 MB in 23.2s



## Step 4 — Final Summary

In [4]:
import os

EXPECTED = [
    ('/content/drive/MyDrive/ResNet50_model_244',  'ResNet50_model_244.pth',  96_457_747),
    ('/content/drive/MyDrive/ResNet101_model_512', 'ResNet101_model_512.pth', 172_739_727),
]

print('── Final Summary ─────────────────────────────────────────────────────')
all_ok = True

for folder, filename, expected_size in EXPECTED:
    fpath = os.path.join(folder, filename)
    print(f'\n📂 {folder}')

    if not os.path.isdir(folder):
        print(f'   ❌ Folder not found')
        all_ok = False
        continue

    if not os.path.isfile(fpath):
        print(f'   ❌ File not found : {filename}')
        all_ok = False
        continue

    actual_size = os.path.getsize(fpath)
    size_str    = f'{actual_size / (1024**2):.2f} MB ({actual_size:,} bytes)'

    if actual_size == expected_size:
        print(f'   ✅ {filename}')
        print(f'      Size : {size_str} — exact match ✔')
    elif actual_size > 0:
        print(f'   ⚠️  {filename}')
        print(f'      Size : {size_str} (expected {expected_size / (1024**2):.2f} MB — may still be valid)')
    else:
        print(f'   ❌ {filename} is empty (0 bytes)')
        all_ok = False

print('\n─────────────────────────────────────────────────────────────────────')
if all_ok:
    print('🏆 All model files are present and verified.')
    print('   You are ready to continue with the class lesson.')
else:
    print('⚠️  One or more files are missing. Please re-run the download cell.')
    print('   If the problem persists, restart the Colab runtime and try again.')

── Final Summary ─────────────────────────────────────────────────────

📂 /content/drive/MyDrive/ResNet50_model_244
   ✅ ResNet50_model_244.pth
      Size : 91.99 MB (96,457,747 bytes) — exact match ✔

📂 /content/drive/MyDrive/ResNet101_model_512
   ✅ ResNet101_model_512.pth
      Size : 164.74 MB (172,739,727 bytes) — exact match ✔

─────────────────────────────────────────────────────────────────────
🏆 All model files are present and verified.
   You are ready to continue with the class lesson.
